In [1]:
import pandas as pd
import numpy as np
import os
import mo_gymnasium as mo_gym
import gymnasium as gym
from env.user_sim import UserSimEnv
import stable_baselines3 as sb3
import torch
from morl_baselines.multi_policy.multi_policy_moqlearning import mp_mo_q_learning
from morl_baselines.multi_policy.pareto_q_learning import pql
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
import value_iteration
import utils
import pickle

In [2]:
MAX_COUNT = 3
NUM_VALS = 3
NUM_STOCHASTIC_STATES = NUM_VALS**3
NUM_STATES = NUM_STOCHASTIC_STATES * (MAX_COUNT+1)**4 
NUM_ACTIONS = 104
NUM_OBJECTIVES = 6

In [3]:
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\'
results_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\results\\'

action_file = 'normalized_challenges.csv'
action_df = pd.read_csv(data_folder + action_file)
action_df['action_id'] = action_df['action_id'] - 1


expert_score_cols = ['score_acceptance', 'score_distraction', 'score_problem_solving', 'score_social_support']
expert_score_matrix = action_df[expert_score_cols].values

category_mapping = {"acceptance": 0, "distraction": 1, "problem_solving": 2, "social_support": 3}
action_df['category_id'] = action_df['category'].map(category_mapping)  
action_categories = action_df['category_id'].values

transition_probs = np.load(data_folder + 'functions\\2\\transition_probs.npy')
reward_matrix = np.load(data_folder + 'functions\\2\\reward_matrix.npy')

NUM_ACTIONS = len(action_df)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = NUM_OBJECTIVES, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=transition_probs, 
                 reward_matrix=reward_matrix, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

In [6]:
# lin_env = mo_gym.wrappers.LinearReward(env, weight=np.array([0.9, 0.1, 0.0]))

# # Run DQN agent!
# agent = sb3.DQN("MultiInputPolicy", lin_env)
# agent.learn(500)

Pareto Q-Learning

In [27]:
num_evals = 30
pop_size = 10
num_steps_per_episode=28
total_steps = 500

eval_env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                    num_objectives = NUM_OBJECTIVES, 
                    expert_score_matrix=expert_score_matrix, 
                    transition_probs=transition_probs, 
                    reward_matrix=reward_matrix, 
                    action_categories=action_categories,
                    MAX_COUNT=MAX_COUNT)

model_name = f'pql_T={total_steps}_nO={NUM_OBJECTIVES}_nA={NUM_ACTIONS}'
filename = model_name
model_file = os.path.join(results_folder, model_name, 'model', filename)

ref_point = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0]) # Emphasize completion probability
pql_agent = pql.PQL(env, ref_point=ref_point, log=False)

front = pql_agent.train(total_timesteps=total_steps, eval_env=eval_env)
print("Training completed. Pareto front:")
print(front)


Training completed. Pareto front:
{(0.8571428571428571, 0.7142857142857143, 0.5714285714285714, 0.25, 1.0, 1.0)}


In [ ]:
data = {
        "non_dominated": pql_agent.non_dominated,
        "avg_reward": pql_agent.avg_reward,
        "env_shape": pql_agent.env_shape,
        "gamma": pql_agent.gamma,
        "ref_point": pql_agent.ref_point
    }
with open(model_file + '.pkl', 'wb') as f:
    pickle.dump(data, f)
print(f"Model saved to {model_file}.pkl")

In [23]:
print(pql_agent.get_local_pcs(0))

{(0.96995328, 0.71959552, 1.2954521600000002, 0.7167513600000002, 1.2954521600000002, 1.2954521600000002), (1.3586285714285715, 1.1474285714285717, 1.6630857142857147, 0.47306666666666675, 1.6192000000000002, 1.7216000000000002), (1.0249801142857142, 0.6222043428571429, 1.2723273142857146, 0.7040128000000001, 1.4692864000000003, 1.5217152000000003), (0.9385937188571432, 1.0961118354285717, 2.0806673554285724, 0.6245492053333335, 2.157715456000001, 2.1912698880000003), (1.1936, 1.2185142857142857, 1.3501714285714288, 0.4312, 1.8192000000000004, 1.9216000000000002), (1.0382043428571428, 0.6068736000000001, 1.3893558857142858, 0.6644864000000001, 1.3668864000000003, 1.4193152000000002), (0.9997107200000003, 1.142848365714286, 2.151371337142858, 0.5735185066666668, 2.2248243200000006, 2.2667673600000007), (0.8918571885714286, 1.2866530742857147, 2.175338788571429, 0.6425497600000001, 2.2667673600000007, 2.2667673600000007), (0.9987657142857143, 0.8285257142857143, 1.5928685714285715, 0.573

Outer-loop MOQ-learning algorithm

In [8]:
num_evals = 10
pop_size = 10
num_steps_per_episode=28
total_steps = 1000

eval_env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                    num_objectives = NUM_OBJECTIVES, 
                    expert_score_matrix=expert_score_matrix, 
                    transition_probs=transition_probs, 
                    reward_matrix=reward_matrix, 
                    action_categories=action_categories,
                    MAX_COUNT=MAX_COUNT)

model_name = f'mpmoq_T={total_steps}_P={pop_size}_nO={NUM_OBJECTIVES}_nA={NUM_ACTIONS}_n_eval={num_evals}'
filename = model_name
model_file = os.path.join(results_folder, model_name, 'model', filename)

mpmoq_agent = mp_mo_q_learning.MPMOQLearning(env, use_gpi_policy=True, weight_selection_algo="gpi-ls", log=False)

ref_point = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0]) 
mpmoq_agent.train(total_timesteps=total_steps, eval_env=eval_env, ref_point=ref_point, num_eval_episodes_for_front=num_evals, timesteps_per_iteration=28)
print("Training completed.")


CCS: [] CCS size: 0
Next weight: [1. 0. 0. 0. 0. 0.]
Adding value: [0.5233 0.5141 0.5522 0.1609 0.6438 0.6438] to CCS.
W_corner: [array([1.1102e-16, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
       0.0000e+00]), array([0., 0., 1., 0., 0., 0.]), array([0., 0., 0., 1., 0., 0.]), array([0., 0., 0., 0., 1., 0.]), array([0., 0., 0., 0., 0., 1.]), array([1., 0., 0., 0., 0., 0.])] W_corner size: 6
CCS: [array([0.5233, 0.5141, 0.5522, 0.1609, 0.6438, 0.6438], dtype=float32)] CCS size: 1
Next weight: [1.1102e-16 1.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00 0.0000e+00]
Adding value: [0.2867 0.3987 0.5377 0.3006 0.5656 0.6209] to CCS.
Value [0.2867 0.3987 0.5377 0.3006 0.5656 0.6209] is dominated. Discarding.
W_corner: [array([1.1102e-16, 1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
       0.0000e+00]), array([0., 0., 1., 0., 0., 0.]), array([0., 0., 0., 1., 0., 0.]), array([0., 0., 0., 0., 1., 0.]), array([0., 0., 0., 0., 0., 1.]), array([1., 0., 0., 0., 0., 0.])] W_corner size: 6
CCS: [

In [12]:
print("Number of policies found:", len(mpmoq_agent.policies))

for i, policy in enumerate(mpmoq_agent.policies):
    print(f"Policy {i}:")
    print(policy.weights)

Number of policies found: 2
Policy 0:
[0. 1. 0. 0. 0. 0.]
Policy 1:
[0. 0. 0. 1. 0. 0.]


In [17]:
mpmoq_agent.env = None
for p in mpmoq_agent.policies:
    p.env = None

os.makedirs(os.path.join(results_folder, model_name, 'model')) 
with open(model_file + ".pkl", "wb") as f:
    pickle.dump(mpmoq_agent, f)


    
print(f"Agent successfully saved to {filename}")

Agent successfully saved to mpmoq_T=1000_P=10_nO=6_nA=104_n_eval=10


### Run simulations

In [24]:
def simulate(env, num_users, policy=None, verbose=False, T=28):
    data_list = []
    for user in range(num_users):
        t = 0
        obs, _ = env.reset()  # later use initial state distribution
        done = False
        while not done and t < T:
            if policy is not None:
                action = policy.eval(obs, w=policy.weights) 
            else:
                action = env.action_space.sample()
            obs_next, rewards, terminated, truncated, info = env.step(action)
            if verbose:
                print(f"Action: {action}, Rewards: {rewards}, Info: {info}")
            user_row = {
                'user': user,
                't': t,
                'action': action,
                'state': obs,
                'next_state': obs_next,
                'counts': obs[3:],
                'rewards': rewards
            }
            data_list.append(user_row)

            done = terminated or truncated
            obs = obs_next
            t += 1
    simulation_results = pd.DataFrame(data_list)
    return simulation_results

In [19]:
NUM_USERS = 1000
NUM_TIMESTEPS = 28

objectives = ["Time Required", "Fun", "Perceived Usefulness", "Expert Usefulness", "Diversity", "Completion Probability"]

In [20]:
policies = mpmoq_agent.policies
print("Number of policies:", len(policies))

Number of policies: 2


In [25]:
save_path = results_folder + model_name + '/simulations/'
save_path = os.path.join(results_folder, model_name, 'simulations')

all_sim_results =[]
if os.path.exists(save_path):
    for i in range(len(policies)):
        file_path = os.path.join(save_path, f'simulation_{i}.pkl')
        df = pd.read_pickle(file_path)
        all_sim_results.append(df)
else:
    os.makedirs(save_path)
    for i, policy in enumerate(policies):
        print(f"Simulating Policy {i}:")
        print(f"  Weights: {policy.weights}")
        simulation_results = simulate(env, NUM_USERS, policy=policy)
        all_sim_results.append(simulation_results)
        save_file = f'simulation_{i}.pkl'
        save_to = os.path.join(save_path, save_file)
        simulation_results.to_pickle(save_to)

Simulating Policy 0:
  Weights: [0. 1. 0. 0. 0. 0.]
Simulating Policy 1:
  Weights: [0. 0. 0. 1. 0. 0.]


In [26]:
random_sim_file = f'random/simulation_random_n={NUM_USERS}.pkl'
random_sim_path = os.path.join(results_folder, random_sim_file)

if os.path.exists(random_sim_path):
    simulation_random = pd.read_pickle(random_sim_path)
else:
    simulation_random = simulate(env, NUM_USERS)
    simulation_random.to_pickle(random_sim_path)

### Analyze and visualize simulations

In [30]:
all_policies = all_sim_results 
policy_names = [f"Policy {i}" for i in range(len(all_sim_results))]

for j in range(NUM_OBJECTIVES):
    print(f"\n--- {objectives[j]} ---")
    for idx, sim_res in enumerate(all_policies):
        raw_rewards = np.stack(sim_res['rewards'])
        rewards = raw_rewards.reshape(NUM_USERS, NUM_TIMESTEPS, NUM_OBJECTIVES)
        
        user_means = np.mean(rewards[:, :, j], axis=1) 
        grand_mean = np.mean(user_means)
        n = len(user_means)
        sem = stats.sem(user_means)
        ci = stats.t.interval(0.95, df=n-1, loc=grand_mean, scale=sem)
        
        print(f"{policy_names[idx]:<15} | Mean: {grand_mean:.4f} | 95% CI: ({ci[0]:.4f}, {ci[1]:.4f})")


--- Time Required ---
Policy 0        | Mean: 0.0690 | 95% CI: (0.0676, 0.0704)
Policy 1        | Mean: 0.0697 | 95% CI: (0.0683, 0.0711)

--- Fun ---
Policy 0        | Mean: 0.0620 | 95% CI: (0.0607, 0.0633)
Policy 1        | Mean: 0.0626 | 95% CI: (0.0614, 0.0639)

--- Perceived Usefulness ---
Policy 0        | Mean: 0.0872 | 95% CI: (0.0854, 0.0890)
Policy 1        | Mean: 0.0878 | 95% CI: (0.0861, 0.0896)

--- Expert Usefulness ---
Policy 0        | Mean: 0.0196 | 95% CI: (0.0192, 0.0201)
Policy 1        | Mean: 0.0199 | 95% CI: (0.0195, 0.0203)

--- Diversity ---
Policy 0        | Mean: 0.0871 | 95% CI: (0.0853, 0.0889)
Policy 1        | Mean: 0.0880 | 95% CI: (0.0863, 0.0897)

--- Completion Probability ---
Policy 0        | Mean: 0.0946 | 95% CI: (0.0926, 0.0966)
Policy 1        | Mean: 0.0956 | 95% CI: (0.0937, 0.0975)


In [31]:
def plot_objective(objective_idx, is_cumulative):
    plt.figure(figsize=(10,6)) 
    obj_name = objectives[objective_idx]
    for idx, sim_res in enumerate(all_policies):
        name = policy_names[idx]
        raw_rewards = np.stack(sim_res['rewards'])
        rewards = raw_rewards.reshape(NUM_USERS, NUM_TIMESTEPS, NUM_OBJECTIVES)
        data = rewards[:, :, objective_idx]
        if is_cumulative:
            data = np.cumsum(data, axis=1)
        mean_rewards = np.mean(data, axis=0)
        style = '--' if "Random" in name else '-'
        plt.plot(mean_rewards, label=name, linestyle=style)

    plt.title(f"Average {obj_name} Over Time {'(Cumulative)' if is_cumulative else ''}")
    plt.xlabel("Time Step")
    plt.ylabel("Mean Reward")
    plt.legend()
    plt.grid(True)
    plt.show()

interact(
    plot_objective, 
    objective_idx=widgets.Dropdown(
        options=[(name, i) for i, name in enumerate(objectives)],
        value=0,
        description='Objective:',
    ),
    is_cumulative=widgets.Checkbox(
        value=True,
        description='Cumulative Sum',
    )
)

interactive(children=(Dropdown(description='Objective:', options=(('Time Required', 0), ('Fun', 1), ('Perceive…

<function __main__.plot_objective(objective_idx, is_cumulative)>

In [32]:
def build_idx_to_count(MAX_COUNT):
    dims = [MAX_COUNT + 1] * 4  # 4 categories
    grid = np.indices(dims)     # shape: (4, ..., ..., ..., ...)
    
    # reshape to (n_c, 4)
    idx_to_count = grid.reshape(4, -1).T
    
    return idx_to_count  # shape (n_c, 4)

def count_to_index(counts, MAX_COUNT):
    base = MAX_COUNT + 1
    return (
        counts[0] * base**3 +
        counts[1] * base**2 +
        counts[2] * base +
        counts[3]
    )

In [33]:
idx_to_count = build_idx_to_count(MAX_COUNT)

In [34]:
def build_next_indices(idx_to_count, MAX_COUNT):
    n_c = idx_to_count.shape[0]
    base = MAX_COUNT + 1
    
    next_indices = np.zeros((4, n_c), dtype=int)  # 4 categories
    
    for k in range(4):
        c_next = idx_to_count.copy()
        
        # increment category k
        c_next[:, k] = np.minimum(c_next[:, k] + 1, MAX_COUNT)
        
        # normalize (your logic)
        min_vals = c_next.min(axis=1, keepdims=True)
        c_next = np.minimum(c_next - min_vals, MAX_COUNT)
        
        # map back to indices
        next_indices[k] = (
            c_next[:, 0] * base**3 +
            c_next[:, 1] * base**2 +
            c_next[:, 2] * base +
            c_next[:, 3]
        )
    
    return next_indices  # shape (4, n_c)

In [35]:
next_indices = build_next_indices(idx_to_count, MAX_COUNT)
print("Next indices shape:", next_indices.shape)  # Should be (4, n_c)

Next indices shape: (4, 256)


In [36]:
def value_iteration_factored(env, weights, action_categories, gamma=0.9, theta=1e-6):
    n_u = 27
    n_c = (MAX_COUNT+1)**4
    V = np.zeros((n_u, n_c)) 
    
    # Pre-calculate scalarized rewards R[u, c, a]
    R = (env.reward_matrix * weights).sum(axis=2).reshape(n_u, n_c, env.nA)
    
    # Pre-extract completion probabilities p_c[u, c, a]
    # (Assuming the last objective is completion probability)
    P_comp = env.reward_matrix[:, :, -1].reshape(n_u, n_c, env.nA)

    while True:
        V_old = V.copy()
        Q = np.zeros((env.nA, n_u, n_c))
        
        for a in range(env.nA):
            # User state transition 
            # P_user_a is (n_u, n_u)
            P_user_a = env.transition_probs[:, a, :] 
            
            # Count state transition 
            # Probability of staying vs probability of incrementing
            p_c = P_comp[:, :, a] # Shape (n_u, n_c)

            k = action_categories[a]  # Category of the current action
            next_indices_k = next_indices[k]  # Shape (n_c,)
            
            # We calculate the expected future value for all user and count states at once
            # Values for staying in same count state
            V_next_stay = P_user_a @ V_old 
            # Values for incrementing count state
            V_next_increment = V_next_stay[:, next_indices_k]
            
            # Combine based on completion probability
            Q[a] = R[:, :, a] + gamma * ((1 - p_c) * V_next_stay + p_c * V_next_increment)

        V = np.max(Q, axis=0)
        
        if np.max(np.abs(V - V_old)) < theta:
            break
            
    return V.flatten(), np.argmax(Q, axis=0).flatten()

In [37]:
# use single-objective RL as baseline to compare
weights = np.array([0.166, 0.166, 0.166, 0.166, 0.166, 0.166])
env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = NUM_OBJECTIVES, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=transition_probs, 
                 reward_matrix=reward_matrix, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

print(f"Reward matrix shape: {env.unwrapped.reward_matrix.shape}")
V, policy_vi = value_iteration_factored(env.unwrapped, weights=weights, action_categories=action_categories)

Reward matrix shape: (6912, 104, 6)


In [38]:
def simulate(env, num_users, policy=None, T=28):
    data_list = []
    for user in range(num_users):
        t = 0
        obs, _ = env.reset()  # later use initial state distribution
        done = False
        while not done and t < T:
            state_idx = utils.state_to_idx(tuple(obs), max_count=MAX_COUNT)
            action = policy[state_idx] if policy is not None else env.action_space.sample()
            obs_next, rewards, terminated, truncated, info = env.step(action)
            user_row = {
                'user': user,
                't': t,
                'action': action,
                'state': obs,
                'next_state': obs_next,
                'counts': obs[3:],
                'rewards': rewards
            }
            data_list.append(user_row)

            done = terminated or truncated
            obs = obs_next
            t += 1
    simulation_results = pd.DataFrame(data_list)
    return simulation_results

In [39]:
simulation_vi = simulate(env, 1000, policy_vi)


In [ ]:
all_policies = all_policies + [simulation_vi]
policy_names = policy_names + ["Policy VI (Equal Weights)"]
interact(
    plot_objective, 
    objective_idx=widgets.Dropdown(
        options=[(name, i) for i, name in enumerate(objectives)],
        value=0,
        description='Objective:',
    ),
    is_cumulative=widgets.Checkbox(
        value=True,
        description='Cumulative Sum',
    )
)

interactive(children=(Dropdown(description='Objective:', options=(('Time Required', 0), ('Fun', 1), ('Perceive…

<function __main__.plot_objective(objective_idx, is_cumulative)>

In [41]:
# use single-objective RL as baseline to compare
weights = [[1.0, 0, 0, 0, 0, 0], [0, 1.0, 0, 0, 0, 0], [0, 0, 1.0, 0, 0, 0], [0, 0, 0, 1.0, 0, 0], [0, 0, 0, 0, 1.0, 0], [0, 0, 0, 0, 0, 1.0]]
env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = NUM_OBJECTIVES, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=transition_probs, 
                 reward_matrix=reward_matrix, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

all_single_policies = []
all_single_simulations = []
for w in weights:
    print(f"VI and simulation on weights: {w}")
    V, policy_vi = value_iteration_factored(env.unwrapped, weights=np.array(w))
    all_single_policies.append(policy_vi)
    print("Policy found, starting simulation")
    simulation_vi = simulate(env, 1000, policy_vi)
    all_single_simulations.append(simulation_vi)

all_single_simulations = all_single_simulations + [simulation_vi, simulation_random]
policy_names = [f"Policy {i}" for i in range(len(all_single_policies))] + ["Equal Weights", "Random Policy"]
all_policies = all_single_simulations
interact(
    plot_objective, 
    objective_idx=widgets.Dropdown(
        options=[(name, i) for i, name in enumerate(objectives)],
        value=0,
        description='Objective:',
    ),
    is_cumulative=widgets.Checkbox(
        value=True,
        description='Cumulative Sum',
    )
)

VI and simulation on weights: [1.0, 0, 0, 0, 0, 0]


TypeError: value_iteration_factored() missing 1 required positional argument: 'action_categories'